In [6]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


import os
for dirname, _, filenames in os.walk('E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-10.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-100.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1000.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1001.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1002.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1003.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1004.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1005.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1006.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1007.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1008.json
E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act\act-print-1009.json
E:/Projects/Data Science/NLP-Projects/LAWBOT

In [ ]:

from pathlib import Path

def new_func():
    return "E:/Projects/Data Science/NLP-Projects/LAWBOT/Data/Act"

INPUT_DIR = Path(new_func())
OUTPUT_DIR = Path("E:/Projects/Data Science/NLP-Projects/LAWBOT/data_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

files = [
    p for p in INPUT_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".json", ".jsonl"}
]

print(f"JSON files found: {len(files)}")

for file in files:
    print(f"{file} | {file.stat().st_size / 1024:.1f} KB")


JSON files found: 0


In [4]:

import json
from collections import Counter

field_counts = Counter()
total_records = 0
parse_errors = []

def read_records(path):
    with path.open("r", encoding="utf-8-sig") as f:
        if path.suffix.lower() == ".jsonl":
            for line_no, line in enumerate(f, 1):
                if line.strip():
                    yield line_no, json.loads(line)
        else:
            data = json.load(f)

            if isinstance(data, list):
                yield from enumerate(data, 1)
            else:
                yield 1, data

for path in files:
    print(f"\n{'=' * 70}\nFILE: {path.name}")

    try:
        for row_no, record in read_records(path):
            total_records += 1

            if isinstance(record, dict):
                field_counts.update(record.keys())

                if row_no <= 3:
                    print(f"\nRecord {row_no}")
                    print("Fields:", list(record.keys()))
                    print(json.dumps(
                        record, ensure_ascii=False, indent=2
                    )[:2500])

            elif row_no <= 3:
                print(f"Record {row_no}: {type(record).__name__}")

    except Exception as exc:
        parse_errors.append({
            "file": str(path),
            "error": repr(exc)
        })

print("\nTOTAL RECORDS:", total_records)
print("\nFIELD FREQUENCY:")
for field, count in field_counts.most_common():
    print(f"{field}: {count}")

print("\nPARSE ERRORS:", parse_errors)



TOTAL RECORDS: 0

FIELD FREQUENCY:

PARSE ERRORS: []


In [ ]:

import hashlib
from collections import Counter

field_presence = Counter()
record_hashes = Counter()
record_types = Counter()
missing_required = Counter()

REQUIRED_CANDIDATES = {
    "id", "act_id", "title", "act_title",
    "text", "content", "sections"
}

record_count = 0

for path in files:
    try:
        for _, record in read_records(path):
            record_count += 1
            record_types[type(record).__name__] += 1

            if not isinstance(record, dict):
                continue

            for key in record:
                field_presence[key] += 1

            present = set(record.keys())

            for key in REQUIRED_CANDIDATES:
                if key in present and record[key] in (None, ""):
                    missing_required[key] += 1

            canonical = json.dumps(
                record,
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":")
            )

            digest = hashlib.sha256(
                canonical.encode("utf-8")
            ).hexdigest()

            record_hashes[digest] += 1

    except Exception as exc:
        print("Could not audit:", path.name, repr(exc))

print("Records:", record_count)
print("Record types:", record_types)
print("\nField coverage:")
for key, count in field_presence.most_common():
    print(f"{key}: {count}/{record_count}")

print("\nEmpty fields:", missing_required)
print(
    "Exact duplicate occurrences:",
    sum(count - 1 for count in record_hashes.values() if count > 1)
)


Records: 1484
Record types: Counter({'dict': 1484})

Field coverage:
act_title: 1484/1484
act_no: 1484/1484
act_year: 1484/1484
publication_date: 1484/1484
sections: 1484/1484
footnotes: 1484/1484
copyright_info: 1484/1484
source_url: 1484/1484
fetch_timestamp: 1484/1484
csv_metadata: 1484/1484
token_count: 1484/1484
language: 1484/1484
government_context: 1484/1484
processing_info: 1484/1484
legal_system_context: 1484/1484

Empty fields: Counter()
Exact duplicate occurrences: 0


In [ ]:

report = {
    "files_found": len(files),
    "total_records": total_records,
    "field_frequency": dict(field_counts),
    "parse_errors": parse_errors,
}

report_path = OUTPUT_DIR / "inspection_report.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Saved:", report_path)
